<a href="https://colab.research.google.com/github/emilsar/NLP-TCGA/blob/main/notebooks/Fall2026/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 — A Model That Learns Its Weights

Labs 2 and 3 used nearest neighbour, which learns nothing: it copies the label of the most
similar report. This lab trains **logistic regression**, which learns one weight per word for
every cancer type. You will compute its loss and its softmax by hand, choose its penalty on the
validation set, test it once, and then look past accuracy: at the types it gets wrong, at how
sure it was, and at the words it relies on.

Lab 3 found that a score measured on hospitals the method has already seen mostly measures
how well it knows their templates. So this lab uses Lab 3's three piles, **split by
hospital**: every score below is about hospitals the model has never seen.

Two kinds of puzzle, as before:

- **Code puzzles**: fill in each `___` and run the cell. An `assert` line tells you
  immediately whether you got it right.
- **THINK puzzles**: no code. Write your answer on the `# ANSWER:` line, in a sentence or two.
  The book section named in the puzzle has what you need.

The book: [Chapter 4, A First Learned Classifier: Logistic Regression](https://obooks.tech/nlp/logistic-regression.html).

## Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

BASE = "https://raw.githubusercontent.com/emilsar/NLP-TCGA/main/data"

# Lab 1 in four lines, plus each report's site code, exactly as in Lab 3.
reports = pd.read_csv(f"{BASE}/corpus/TCGA_Reports.csv")
labels = pd.read_csv(f"{BASE}/cancer_type/tcga_patient_to_cancer_type.csv")
reports["patient_id"] = reports["patient_filename"].apply(lambda x: x.split(".")[0])
reports["cancer_type"] = labels.set_index("patient_id").loc[reports["patient_id"], "cancer_type"].values
reports["site"] = reports["patient_id"].apply(lambda b: b.split("-")[1])

# Lab 3's three piles, split by hospital: no site appears in more than one pile.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
rows_a, rows_b = next(splitter.split(reports, groups=reports["site"]))
train_all, test = reports.iloc[rows_a], reports.iloc[rows_b]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
rows_a, rows_b = next(splitter.split(train_all, groups=train_all["site"]))
train, val = train_all.iloc[rows_a], train_all.iloc[rows_b]

print(len(train), "train,", len(val), "validation,", len(test), "test reports")

## How wrong is a prediction?

A model is trained by making its **loss** as small as possible. For logistic regression the loss
of one prediction is minus the log of the probability the model gave to the right answer.

In [ ]:
# PUZZLE 1 — the cross-entropy loss, from the probability given to the right answer.
# Chapter 4, "Cross-entropy loss: how wrong is a prediction?"; np.log is in Appendix B,
# "Functions over a whole array".

def loss(p_right):
    return -np.___(p_right)

for p in [0.99, 0.9, 0.6, 0.4, 0.1, 0.01]:
    print(f"probability given to the right answer {p:<5}  loss {loss(p):.2f}")

assert round(loss(0.5), 3) == 0.693

In [ ]:
# PUZZLE 2 — THINK
# A prediction of 0.6 and a prediction of 0.99 for the right answer are both correct, so
# accuracy scores them the same. Their losses are 0.51 and 0.01. Why does training need a
# score that tells them apart?
# Chapter 4, "Cross-entropy loss: how wrong is a prediction?"
#
# ANSWER: ___

## From scores to probabilities

With more than two types, the model gives each type a score, its **logit**, and the
**softmax** turns the scores into probabilities that add up to 1. These are the three logits
from the chapter's six-report example.

In [ ]:
# PUZZLE 3 — the softmax: raise e to each logit, then divide by the total.
# Chapter 4, "Three classes: the six reports".

def softmax(z):
    e = np.exp(z)
    return e / e.___()

logits = np.array([-0.4179, 0.4963, -0.0784])
print("softmax:              ", softmax(logits).round(3))
print("add 1 to every logit: ", softmax(logits + 1).round(3))

assert round(softmax(logits).sum(), 6) == 1
assert np.allclose(softmax(logits), softmax(logits + 1))

In [ ]:
# PUZZLE 4 — THINK
# Adding 1 to every logit changed nothing. So what about the logits does the softmax actually
# use?
# Chapter 4, "Three classes: the six reports" (the softmax widget).
#
# ANSWER: ___

## Choosing the penalty

On the real corpus the features are word **presence** (1 if a word occurs in a report at all,
0 if not), which won in Chapter 4's table. The penalty strength `C` is chosen on the validation
set, as *k* was in Chapter 2. This cell takes about half a minute.

In [ ]:
# PUZZLE 5 — two blanks: word presence instead of counts, then the penalty strength being tried.
# Chapter 4, "Keeping the weights in check: regularization" and "Logistic regression on the
# real corpus".

vec = CountVectorizer(binary=___)
X_train = vec.fit_transform(train["text"])
X_val = vec.transform(val["text"])

scores = {}
for C in [0.01, 1, 100]:
    model = LogisticRegression(C=___, max_iter=3000).fit(X_train, train["cancer_type"])
    scores[C] = (model.predict(X_val) == val["cancer_type"]).mean()
    print(f"C = {C:<5} validation accuracy {scores[C]:.3f}   largest weight {np.abs(model.coef_).max():.2f}")

assert max(scores, key=scores.get) == 1

In [ ]:
# PUZZLE 6 — THINK
# As C grows, the largest weight grows. What would happen to the weights with no penalty at
# all, given that 5,489 reports and tens of thousands of word columns can be separated
# perfectly? Why is C chosen on the validation set and not on the test set?
# Chapter 4, "Keeping the weights in check: regularization".
#
# ANSWER: ___

## Open the test set, once

`C` = 1 is chosen. As in Chapter 2, the model is fitted again on train and validation together,
and run on the test set once.

In [ ]:
# PUZZLE 7 — the pile to fit on: train and validation together.
# Chapter 2, "What to tune".

vec = CountVectorizer(binary=True)
X_all = vec.fit_transform(___["text"])
X_test = vec.transform(test["text"])
model = LogisticRegression(C=1, max_iter=3000).fit(X_all, train_all["cancer_type"])

pred = model.predict(X_test)
accuracy = (pred == test["cancer_type"]).mean()

assert round(accuracy, 3) == 0.846
print("Test accuracy, logistic regression:", round(accuracy, 3))
print("Majority-class baseline:           ", round(test["cancer_type"].value_counts(normalize=True).max(), 3))

## Beyond accuracy

Accuracy is one number for 32 types of very different sizes. The **macro** F1 averages the F1
of every type with an equal vote, however small the type.

Splitting by hospital leaves four rare types with no test reports at all (Lab 3). A type
the model predicts but which has no test reports, such as KICH, counts in the macro average
with an F1 of 0; `zero_division=0` says so instead of printing a warning.

In [ ]:
# PUZZLE 8 — the kind of average that gives every type an equal vote.
# Chapter 4, "F1, and averaging over classes".
from sklearn.metrics import classification_report, f1_score

macro = f1_score(test["cancer_type"], pred, average="___", zero_division=0)

assert round(macro, 3) == 0.744
print(classification_report(test["cancer_type"], pred, digits=2, zero_division=0))

Two rows of the report stand out.

**ACC** (adrenocortical carcinoma) scores 0: 80 of its 90 reports come from a single hospital,
site `OR`, and that hospital landed in the test pile. The model trained on the other 10. This is
the price of splitting by hospital that Lab 3 met with thymoma, and it is also a fair warning:
a hospital that sends many reports of one type can make that type look easy or impossible.

**Colon (COAD) and rectum (READ)**, one of the three pairs Chapter 4 found hardest. Just run it.

In [ ]:
pair = test["cancer_type"].isin(["COAD", "READ"]).values
pd.crosstab(test["cancer_type"].values[pair], pred[pair], rownames=["true"], colnames=["predicted"])

In [ ]:
# PUZZLE 9 — THINK
# In the report above, READ has precision 0.16 and recall 1.00. Say in plain words what each
# number means for READ, and use the table to say where the wrong "rectum" answers came from.
# Chapter 4, "Precision and recall" and "The real report".
#
# ANSWER: ___

## How sure was it?

`predict_proba` gives every report one probability per type. The largest one in each row is
the probability of the type the model predicted: `.max(axis=1)` takes the largest number in
each row.

In [ ]:
# PUZZLE 10 — the method that gives the probabilities.
# Chapter 4, "Three classes: the six reports".

confidence = model.___(X_test).max(axis=1)
wrong = (pred != test["cancer_type"]).values

sure_but_wrong = (confidence[wrong] >= 0.9).sum()

assert wrong.sum() == 335 and sure_but_wrong == 32
print("median confidence when right:", round(np.median(confidence[~wrong]), 3))
print("median confidence when wrong:", round(np.median(confidence[wrong]), 3))
print(sure_but_wrong, "of the", wrong.sum(), "mistakes were made with a probability of 0.9 or more")

In [ ]:
# PUZZLE 11 — THINK
# A hospital suggests: "Use the model's answer whenever it is at least 90% sure, and send the
# rest to a pathologist." What does that rule gain, and what does it not guarantee?
# Chapter 4, "Confidence is not correctness".
#
# ANSWER: ___

## What the model learned

Each type has one weight per word, in `model.coef_`. A large positive weight pushes a report
toward that type.

In [ ]:
# PUZZLE 12 — the positions of the 8 largest READ weights, largest first.
# Appendix B, "The top-N idiom".

words = vec.get_feature_names_out()
read = list(model.classes_).index("READ")

top = model.coef_[read].argsort()[___][::-1]

assert words[top[0]] == "rectum"
print(list(words[top]))

In [ ]:
# PUZZLE 13 — THINK
# Among READ's top words are "g2" (a grade written as a code) and "ro" (the scanner's
# misreading of "R0"). Each appears in reports from only two to five hospitals. Which of the
# eight words describe the disease, and which describe who wrote or scanned the report? Why
# is a model that leans on the second kind risky at a hospital it has never seen?
# Chapter 4, "What the model learned" and "Hospitals it has never seen".
#
# ANSWER: ___

## What you did

- Split every pile by hospital, as in Lab 3, so every score is about hospitals the model has
  never seen.
- Computed the cross-entropy loss and the softmax by hand.
- Chose the penalty `C` on the validation set, and saw that a larger `C` lets the weights grow.
- Tested once: **0.846**, against Lab 3's best of 0.489 for nearest neighbour on the same test
  hospitals, and a baseline of 0.108.
- Looked past accuracy: a macro F1 of 0.744, one type (ACC) lost because one hospital held
  nearly all of it, and colon against rectum still confused.
- Found that 32 of the 335 mistakes were made with 90% confidence or more.
- Read the model's weights, and found scanning errors and template codes among the anatomy.

Chapter 4 also quotes 0.964 for a split made report by report, where the test hospitals have
reports in training too. The twelve points between the two numbers are what knowing a
hospital's template is worth. A model that learns one weight per word, shared across every
hospital, keeps most of its score anyway: it ends up relying mostly on the words of the disease.